In [0]:
from pyspark.sql.types import StringType, DateType, IntegerType
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.crm_sales_details")
display(df)

In [0]:
df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()

In [0]:
df.filter(F.col("sls_price") < 0).show()

In [0]:
df.filter(F.col("sls_quantity") < 0).show()

In [0]:
df = (
    df.withColumn(
        "sls_sales",
        F.when(F.col("sls_sales").isNull() | (F.col("sls_sales") < 0) , F.col("sls_quantity") * F.abs(F.col("sls_price")))
        .otherwise(F.col("sls_sales"))
    )
    .withColumn(
        "sls_price",
        F.when(F.col("sls_price").isNull(), F.try_divide(F.col("sls_sales"), F.col("sls_quantity")))
        .when(F.col("sls_price") < 0, F.abs(F.col("sls_price")))
        .otherwise(F.col("sls_price"))
    )
)

In [0]:
df.filter(F.col("sls_price") < 0).show()

In [0]:
df.printSchema()

In [0]:
df = df.dropDuplicates()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
date_cols = ['sls_order_dt', 'sls_ship_dt', 'sls_due_dt']
for date_col in date_cols:
    df = df.withColumn(date_col, F.try_to_date(F.col(date_col), 'yyyyMMdd'))


In [0]:
display(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_sales